# Tasty Bytes Sales Forecast – Remote Development

This notebook runs against a Snowflake remote development environment (SPCS-backed) reached over Remote-SSH from VS Code or Cursor. It uses the **Snowflake Kernel (Python + SQL)**, the same kernel that powers Snowflake Notebooks in Workspaces, so Python and SQL cells run side by side without switching kernels.

Sequence:
1. Confirm the raw Tasty Bytes tables loaded correctly.
2. Aggregate orders to a daily-per-location grain.
3. Join weather from the Marketplace share.
4. Feature-engineer with help from CoCo.
5. Land a feature table.
6. Train XGBoost and log to the Model Registry.
7. Chart actual vs predicted sales for the holdout week.

## 0. Set the session context

SQL cells run against your Snowflake account and need an active **query warehouse**. Set the warehouse and a default database once here, before running anything else.

In [ ]:
USE WAREHOUSE tb_de_wh;
USE DATABASE tb_101;

## 1. Confirm the raw tables loaded

In [ ]:
SELECT 'order_header' AS tbl, COUNT(*) AS row_count FROM tb_101.raw_pos.order_header
UNION ALL SELECT 'order_detail', COUNT(*) FROM tb_101.raw_pos.order_detail
UNION ALL SELECT 'location',     COUNT(*) FROM tb_101.raw_pos.location
UNION ALL SELECT 'truck',        COUNT(*) FROM tb_101.raw_pos.truck
UNION ALL SELECT 'menu',         COUNT(*) FROM tb_101.raw_pos.menu
UNION ALL SELECT 'customer_loyalty', COUNT(*) FROM tb_101.raw_customer.customer_loyalty
ORDER BY 2 DESC;

## 2. Aggregate to daily-per-location grain

We roll ~1B `order_detail` line items up to one row per (`date`, `location_id`, `city`) via the `harmonized.orders_v` view. Snowflake does this in seconds on the Large warehouse.

In [ ]:
CREATE OR REPLACE TABLE tb_101.ml.daily_sales AS
SELECT
    DATE(order_ts)           AS date,
    location_id,
    ANY_VALUE(location_city) AS city,
    ANY_VALUE(location_region) AS region,
    ANY_VALUE(country)       AS country,
    SUM(price)               AS daily_sales,
    COUNT(DISTINCT order_id) AS order_count
FROM tb_101.harmonized.orders_v
GROUP BY 1, 2;

SELECT COUNT(*) AS rows_in_daily_sales FROM tb_101.ml.daily_sales;

## 3. Join Marketplace weather

`setup.sql` installed the Pelmorex Weather Source: Frostbyte share as the `frostbyte_weathersource` database. Its `history_day` view is keyed at **postal-code + date** grain, so joining directly on city would fan every city out to all of its postal codes (Denver alone has ~74) and explode the row count. We first pre-aggregate weather to one row per city per date, then join on `date` + `city` and convert to metric. We also bump the warehouse to `MEDIUM` for this step (the shared weather view is large and slow to scan on `XSMALL`) and size it back down right after.

In [ ]:
-- Confirm the weather share is present (installed by setup.sql)
SHOW DATABASES LIKE 'FROSTBYTE_WEATHERSOURCE';

In [ ]:
ALTER WAREHOUSE tb_de_wh SET WAREHOUSE_SIZE = 'MEDIUM';

CREATE OR REPLACE TABLE tb_101.ml.daily_sales_weather AS
WITH weather_by_city AS (
    SELECT
        date_valid_std,
        UPPER(city_name)              AS city_upper,
        AVG(avg_temperature_air_2m_f) AS avg_temperature_air_2m_f,
        AVG(tot_precipitation_in)     AS tot_precipitation_in,
        AVG(avg_wind_speed_100m_mph)  AS avg_wind_speed_100m_mph
    FROM frostbyte_weathersource.onpoint_id.history_day
    WHERE date_valid_std BETWEEN (SELECT MIN(date) FROM tb_101.ml.daily_sales)
                             AND (SELECT MAX(date) FROM tb_101.ml.daily_sales)
      AND UPPER(city_name) IN (SELECT DISTINCT UPPER(city) FROM tb_101.ml.daily_sales)
    GROUP BY date_valid_std, UPPER(city_name)
)
SELECT
    ds.date,
    ds.location_id,
    ds.city,
    ds.daily_sales,
    ds.order_count,
    w.avg_temperature_air_2m_f                     AS temp_f,
    (w.avg_temperature_air_2m_f - 32.0) * 5.0/9.0  AS temp_c,
    w.tot_precipitation_in * 25.4                  AS precip_mm,
    w.avg_wind_speed_100m_mph * 1.60934            AS wind_kph
FROM tb_101.ml.daily_sales ds
JOIN weather_by_city w
  ON w.date_valid_std = ds.date
 AND w.city_upper = UPPER(ds.city)
WHERE ds.daily_sales IS NOT NULL;

ALTER WAREHOUSE tb_de_wh SET WAREHOUSE_SIZE = 'XSMALL';

SELECT COUNT(*) AS joined_rows FROM tb_101.ml.daily_sales_weather;

## 4. Feature-engineer with CoCo

We need calendar, lag, and rolling features on the joined data – the standard patterns for time-series forecasting. Rather than hand-write them, let CoCo generate them: open the CoCo panel in the remote window and try prompts like "Add day-of-week, month, and is_weekend features", "Add lag-1 and lag-7 daily-sales features per location_id", and "Add a 7-day rolling mean of precip_mm per location_id".

By default CoCo shows the generated code in its panel for you to read or copy – it won't change the notebook on its own. If you attach this notebook as context with `@`, CoCo will instead try to apply its code as edits to the file. Either way you don't need to accept any edits: the two cells below are already provided – one defines the transforms, one applies them. Run them as-is to continue, and use CoCo alongside to see how you'd generate them yourself.

In [ ]:
import pandas as pd

def add_calendar_features(df, date_col='date'):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df['day_of_week'] = df[date_col].dt.dayofweek
    df['month'] = df[date_col].dt.month
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    return df

def add_lag_features(df, group_cols, target_col='daily_sales', lags=(1, 7)):
    df = df.copy().sort_values(group_cols + ['date'])
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = df.groupby(group_cols)[target_col].shift(lag)
    return df

def add_rolling_precip(df, group_cols, precip_col='precip_mm', window=7):
    df = df.copy().sort_values(group_cols + ['date'])
    df[f'{precip_col}_roll_{window}'] = (
        df.groupby(group_cols)[precip_col]
        .rolling(window=window, min_periods=1)
        .mean()
        .reset_index(level=list(range(len(group_cols))), drop=True)
    )
    return df

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
raw = session.table('tb_101.ml.daily_sales_weather').to_pandas()
raw.columns = [c.lower() for c in raw.columns]

features_df = (
    raw
    .pipe(add_calendar_features)
    .pipe(add_rolling_precip, group_cols=['location_id'])
    .pipe(add_lag_features, group_cols=['location_id'])
    .dropna()
    .reset_index(drop=True)
)
features_df.head()

## 5. Land the feature table

We push the engineered DataFrame back to Snowflake as `tb_101.ml.feature_table` so `train.py` can pick it up.

In [ ]:
session.write_pandas(
    features_df,
    table_name='FEATURE_TABLE',
    database='TB_101',
    schema='ML',
    auto_create_table=True,
    overwrite=True,
)
session.table('tb_101.ml.feature_table').count()

## 6. Train + log to the Model Registry

`train.py` is a plain Python script; the remote environment runs `.py` files the same way it runs notebook cells. You can call it from a terminal (`python train.py --source-table tb_101.ml.feature_table --database tb_101 --schema ml`) or from a cell as below.

In [ ]:
!python /mnt/pd0/sfguide-getting-started-with-remote-development-vscode-extension/train.py \
    --source-table tb_101.ml.feature_table \
    --database tb_101 \
    --schema ml \
    --model-name tb_sales_forecaster \
    --version v1

## 7. Chart actual vs predicted for one location

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name='TB_101', schema_name='ML')
model_ref = registry.get_model('tb_sales_forecaster').version('v1')

holdout = features_df[features_df['date'] > features_df['date'].max() - pd.Timedelta(days=7)].copy()
one_loc = holdout[holdout['location_id'] == holdout['location_id'].value_counts().idxmax()].copy()

FEATURE_COLS = [
    'day_of_week', 'month', 'is_weekend',
    'temp_c', 'precip_mm', 'wind_kph',
    'precip_mm_roll_7', 'daily_sales_lag_1', 'daily_sales_lag_7',
]
# model_ref.run returns a DataFrame; the prediction is the last column.
one_loc['predicted'] = model_ref.run(one_loc[FEATURE_COLS]).iloc[:, -1].to_numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(one_loc['date'], one_loc['daily_sales'], marker='o', label='Actual')
ax.plot(one_loc['date'], one_loc['predicted'], marker='x', label='Predicted')
ax.set_title(f"Location {int(one_loc['location_id'].iloc[0])} – actual vs predicted (holdout week)")
ax.set_ylabel('Daily sales (USD)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()